In [1]:
import pandas as pd
import ast
import os
import warnings

warnings.filterwarnings("ignore")

def extract_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
import time
import glob
from tqdm import tqdm
from Bio import SeqIO
import multiprocessing
import subprocess

def max_dict(dic):
    max_num = None
    for key in dic:
        try:
            int(max_num)
        except:
            max_num = dic[key]
        if dic[key] >= max_num:
            max_key = key
            max_num = dic[key]
    return max_key, max_num

def find_oriv(acc_n, max_key, result_dir, que):
    handle = open(f'/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/data/{acc_n}/genomic.gbff')
    acc_record = SeqIO.parse(handle, 'genbank')
    for seq_record in acc_record:
        if seq_record.id == max_key or glob.glob(f'{result_dir}/{acc_n}/{seq_record.id}/selected_ori_regions.csv'):
            continue
        else:
            os.makedirs(f'{result_dir}/{acc_n}', exist_ok=True)
            file_name = f'{result_dir}/{acc_n}/{seq_record.id}.fasta'
            ncl_file = open(file_name, 'w+')
            seq_record.description = ''
            SeqIO.write(seq_record, ncl_file, 'fasta')
            ncl_file.close()
            output_dir = f"/app/data/{seq_record.id}"
            
            cmd = [
                "docker", "run", "--rm",
                "-v", f"{result_dir}/{acc_n}:/app/data",
                "-v", f"{result_dir}/{acc_n}:/app/input",
                "orivfinder-ready",
                "python", "-W", "ignore", "oriVfinder.py",
                "--fasta", f"/app/input/{seq_record.id}.fasta",
                "--output_dir", output_dir
            ]
            try:
                subprocess.run(
                    cmd,
                    check=True,
                    stdout=subprocess.DEVNULL,
                    stderr=subprocess.DEVNULL
                )
            except:
                pass
            os.system(f'rm {file_name}')
    que.put(1)


for genus_name in keep_genus:
    org_data_n = all_data[all_data['genus_clean'].str.contains(genus_name, na=False)].reset_index(drop=True)
    result_dir = f'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}/Ori_finder/orivfinder'

    manager = multiprocessing.Manager()
    que = manager.Queue()
    
    par = 32
    tot = len(org_data_n)
    pool = multiprocessing.Pool(par)

    for i in org_data_n.index:
        acc_n = org_data_n['accession'][i]
        chr_data = ast.literal_eval(org_data_n['chromosome contigs'][i])
        pla_data = ast.literal_eval(org_data_n['plasmid contigs'][i])
        merge_data = chr_data | pla_data
        max_key, max_num = max_dict(merge_data)
        pool.apply_async(find_oriv, (acc_n, max_key, result_dir, que))

    pool.close()
    
    count = 0
    with tqdm(total = tot, desc=f'{genus_name}({tot})', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        while True:
            time.sleep(0.001)
            if not que.empty():
                value = que.get(True)
                count += 1
                pbar.update(1)
                if count == tot:
                    break
            else:
                continue
     
    pool.join()

Escherichia(4204): 100%|████████████████████████████████████████| 4.20k/4.20k [03:43<00:00, 18.8B/s]
Klebsiella(3554): 100%|███████████████████████████████████████| 3.55k/3.55k [9:40:12<00:00, 9.80s/B]
Staphylococcus(2423): 100%|███████████████████████████████████| 2.42k/2.42k [1:44:31<00:00, 2.59s/B]
Pseudomonas(2343): 100%|████████████████████████████████████████| 2.34k/2.34k [58:34<00:00, 1.50s/B]
Bacillus(1976): 100%|█████████████████████████████████████████| 1.98k/1.98k [2:25:49<00:00, 4.43s/B]
Salmonella(1853): 100%|███████████████████████████████████████| 1.85k/1.85k [2:06:04<00:00, 4.08s/B]
Streptococcus(1599): 100%|██████████████████████████████████████| 1.60k/1.60k [09:20<00:00, 2.85B/s]
Streptomyces(1359): 100%|█████████████████████████████████████| 1.36k/1.36k [1:58:20<00:00, 5.23s/B]
Acinetobacter(1234): 100%|████████████████████████████████████| 1.23k/1.23k [1:50:39<00:00, 5.38s/B]
Helicobacter(416): 100%|████████████████████████████████████████████| 416/416 [08:13<00:00,

In [2]:
# with error 
import time
from tqdm import tqdm
from Bio import SeqIO
import multiprocessing
import subprocess

def max_dict(dic):
    max_num = None
    for key in dic:
        try:
            int(max_num)
        except:
            max_num = dic[key]
        if dic[key] >= max_num:
            max_key = key
            max_num = dic[key]
    return max_key, max_num

def find_oriv(acc_n, max_key, result_dir, que):
    handle = open(f'/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/data/{acc_n}/genomic.gbff')
    acc_record = SeqIO.parse(handle, 'genbank')
    for seq_record in acc_record:
        if seq_record.id == max_key:
            continue
        else:
            os.makedirs(f'{result_dir}/{acc_n}', exist_ok=True)
            file_name = f'{result_dir}/{acc_n}/{seq_record.id}.fasta'
            ncl_file = open(file_name, 'w+')
            seq_record.description = ''
            SeqIO.write(seq_record, ncl_file, 'fasta')
            ncl_file.close()
            output_dir = f"/app/data/{seq_record.id}"
            
            cmd = [
                "docker", "run", "--rm",
                "-v", f"{result_dir}/{acc_n}:/app/data",
                "-v", f"{result_dir}/{acc_n}:/app/input",
                "orivfinder-ready",
                "python", "-W", "ignore", "oriVfinder.py",
                "--fasta", f"/app/input/{seq_record.id}.fasta",
                "--output_dir", output_dir
            ]
            subprocess.run(
                cmd,
                check=True,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL
            )
            os.system(f'rm {file_name}')
    que.put(1)


for genus_name in keep_genus:
    org_data_n = all_data[all_data['genus_clean'].str.contains(genus_name, na=False)].reset_index(drop=True)
    result_dir = f'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}/Ori_finder/orivfinder'
    
    manager = multiprocessing.Manager()
    que = manager.Queue()
    
    par = 32
    tot = len(org_data_n)
    pool = multiprocessing.Pool(par)

    for i in org_data_n.index:
        acc_n = org_data_n['accession'][i]
        chr_data = ast.literal_eval(org_data_n['chromosome contigs'][i])
        pla_data = ast.literal_eval(org_data_n['plasmid contigs'][i])
        merge_data = chr_data | pla_data
        max_key, max_num = max_dict(merge_data)
        pool.apply_async(find_oriv, (acc_n, max_key, result_dir, que))

    pool.close()
    
    count = 0
    with tqdm(total = tot, desc=f'{genus_name}({tot})', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        while True:
            time.sleep(0.001)
            if not que.empty():
                value = que.get(True)
                count += 1
                pbar.update(1)
                if count == tot:
                    break
            else:
                continue
    
    pool.join()

Escherichia(4204): 100%|████████████████████████████████████▉| 4.20k/4.20k [25:30:35<01:49, 21.9s/B]


KeyboardInterrupt: 